# Open LLMs available for commercial use
https://github.com/eugeneyan/open-llms

In [1]:
# Install dependencies as needed:
!pip install kagglehub[pandas-datasets]
!pip install emoji

# ==============================
# Core ML / DL
# ==============================
!pip install torch torchvision torchaudio --quiet
!pip install pytorch-lightning --quiet

# ==============================
# NLP / Transformers
# ==============================
!pip install transformers datasets --quiet

# ==============================
# Classic ML 
# ==============================
!pip install scikit-learn --quiet

# ==============================
# Text processing (your pipeline deps)
# ==============================
!pip install nltk symspellpy emot gensim --quiet

# ==============================
# Visualization
# ==============================
!pip install matplotlib seaborn wordcloud --quiet

# ==============================
# (Optional but useful)
# ==============================
!pip install tqdm --quiet
!pip install seaborn plotly


import nltk
nltk.download("stopwords")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 35.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 7.1 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from itertools import chain
import re
import time
import tracemalloc
from itertools import chain
from collections import Counter

import nltk
from nltk.corpus import stopwords

from symspellpy.symspellpy import SymSpell, Verbosity
import pkg_resources

from emot.emo_unicode import EMOTICONS_EMO

from sklearn.preprocessing import LabelEncoder
import pandas as pd
import pytorch_lightning as pl
import torch
print(torch.cuda.is_available())  # should be True
from transformers import AutoTokenizer, AutoModel

/tmp/ipykernel_55/361885133.py:14: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


False


In [3]:

file_path = "IMDB Dataset.csv"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews", 
    file_path,
)

print("First 5 records:")
print(df.head())

First 5 records:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [4]:

# ==============================
# Constants (UNCHANGED)
# ==============================
keep_single = "!?"
keep_sequences = r"(\.\.\.|!!+|\?\?+|\?!+|!\?+)"
emoticon_pattern = r'[:;=8][\-o\*\']?[\)\]\(\[dDpP/\:\}\{@\|\\]'

keep_words = {"not", "no", "nor"}
RARE_THRESHOLD = 7

chat_words = {
    "fyi": "for your information",
    "asap": "as soon as possible",
    "brb": "be right back",
    "btw": "by the way",
    "omg": "oh my god",
    "imo": "in my opinion",
    "lol": "laugh out loud",
    "ttyl": "talk to you later",
    "gtg": "got to go",
    "ttyt": "talk to you tomorrow",
    "idk": "i don't know",
    "tmi": "too much information",
    "imho": "in my humble opinion",
    "icymi": "in case you missed it",
    "afaik": "as far as i know",
    "faq": "frequently asked questions",
    "tgif": "thank god it's friday",
    "fya": "for your action",
    "smh": "shaking my head",
    "ftw": "for the win",
    "btwn": "between",
    "np": "no problem",
    "yolo": "you only live once",
    "omfg": "oh my freaking god",
    "rofl": "rolling on the floor laughing",
    "lmao": "laughing my ass off",
    "bff": "best friends forever",
    "jk": "just kidding",
    "idc": "i don't care",
    "tbh": "to be honest",
    "yk": "you know"
}

# ==============================
# Init
# ==============================
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english')) - keep_words

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

# ==============================
# Precompiled regex (FASTER)
# ==============================
EMO_PATTERN = re.compile("|".join(map(re.escape, EMOTICONS_EMO.keys())))
CHAT_PATTERN = re.compile(r'\b(' + '|'.join(chat_words.keys()) + r')\b', flags=re.I)
TOKEN_PATTERN = re.compile(r"\w+|[!?]+|\.\.\.|[:;=8][\-o\*\']?[\)\]\(\[dDpP/\:\}\{@\|\\]")

# ==============================
# Monitor decorator
# ==============================
def monitor(func):
    def wrapper(*args, **kwargs):
        tracemalloc.reset_peak()  # reset peak for this function
        start_time = time.time()
        current_before, _ = tracemalloc.get_traced_memory()

        result = func(*args, **kwargs)

        end_time = time.time()
        current_after, peak_after = tracemalloc.get_traced_memory()

        print(f"{func.__name__}:")
        print(f"  Time: {end_time - start_time:.2f}s")
        print(f"  Memory Δ: {(current_after - current_before)/1e6:.2f} MB")
        print(f"  Peak usage: {peak_after/1e6:.2f} MB")

        # -----------------------------
        # Check unique tokens if applicable
        # -----------------------------
        if isinstance(result, pd.Series) and all(isinstance(t, list) for t in result):
            # flatten safely
            all_tokens = list(chain.from_iterable(result))
            unique_tokens = set(all_tokens)
            print(f"  Total tokens: {len(all_tokens)}")
            print(f"  Unique tokens: {len(unique_tokens)}")

        print("")  # newline
        return result
    return wrapper
# ==============================
# Cleaning
# ==============================
@monitor
def normalize_numbers_series(series):
    return series.apply(lambda text: re.sub(r'\b\d+(\.\d+)?\b', "<NUM>", text))

def normalize_punct(seq):
    if re.fullmatch(r'!{2,}', seq): return "!!+"
    elif re.fullmatch(r'\?{2,}', seq): return "??+"
    elif re.fullmatch(r'\.{2,}', seq): return "..."
    elif re.fullmatch(r'(!\?|\?\!){2,}', seq): return seq[:2]
    return seq

@monitor
def clean_review_series(series):
    def clean(text):
        text = re.sub(
            rf"(?!{keep_single}|{keep_sequences}|{emoticon_pattern})[^\w\s]",
            " ",
            text
        )
        return re.sub(r'(\.\.\.|!!+|\?\?+|!\?+|\?\!+)',
                      lambda m: normalize_punct(m.group(0)), text)
    return series.apply(clean)

@monitor
def normalize_emoticons_series(series):
    return series.apply(lambda text: EMO_PATTERN.sub(lambda m: EMOTICONS_EMO[m.group()], text))

@monitor
def replace_chat_words_series(series):
    return series.apply(lambda text: CHAT_PATTERN.sub(lambda x: chat_words[x.group().lower()], text))

# ==============================
# Tokenization
# ==============================
@monitor
def tokenize_series(series):
    return series.apply(lambda text: TOKEN_PATTERN.findall(text))

@monitor
def filter_single_letters_series(series):
    return series.apply(lambda tokens: [w for w in tokens if len(w) > 1 or w.lower() in {"i", "a"}])

# ==============================
# Spell correction (OPTIMIZED)
# ==============================
def is_elongated(word):
    return any(word[i] == word[i+1] == word[i+2] for i in range(len(word)-2))

def should_skip_spellcheck(word, freq_dict):
    # Skip conditions (FAST FILTER)
    if not word.isalpha():
        return True
    if len(word) < 3:
        return True
    if is_elongated(word):
        return True
    # Likely names / rare weird tokens
    if freq_dict.get(word, 0) <= 2:
        return True
    return False

@monitor
def build_correction_map(tokens_series):
    all_tokens = list(chain.from_iterable(tokens_series))
    freq_dict = Counter(all_tokens)
    unique_words = set(all_tokens)

    correction_map = {}

    for word in unique_words:
        if should_skip_spellcheck(word, freq_dict):
            correction_map[word] = word
            continue

        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        correction_map[word] = suggestions[0].term if suggestions else word

    return correction_map

@monitor
def apply_corrections_series(series, correction_map):
    return series.apply(lambda tokens: [correction_map[w] for w in tokens])

# ==============================
# Stopwords + Rare
# ==============================
@monitor
def remove_stopwords_series(series):
    return series.apply(lambda tokens: [w for w in tokens if w.lower() not in stop_words])

@monitor
def build_rare_tokens(tokens_series):
    all_tokens = list(chain.from_iterable(tokens_series))
    freq = Counter(all_tokens)
    return {w for w, c in freq.items() if c <= RARE_THRESHOLD}

@monitor
def replace_rare_series(series, rare_tokens):
    return series.apply(lambda tokens: [t if t not in rare_tokens else "<UNK>" for t in tokens])

# ==============================
# PIPELINE
# ==============================
def process_dataframe(df):

    tracemalloc.start()
    df = df.copy()

    # Basic cleaning
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)

    df["review"] = df["review"].str.replace(r"<br\s*/?>", " ", regex=True)\
                                 .str.replace(r"<.*?>", "", regex=True)

    df["review"] = df["review"].str.replace(r'https?://\S+|www\.\S+', '', regex=True)

    df["review"] = normalize_numbers_series(df["review"])
    df["review"] = df["review"].str.casefold()

    # Text cleaning
    df["review"] = clean_review_series(df["review"])
    df["review"] = normalize_emoticons_series(df["review"])
    df["review"] = replace_chat_words_series(df["review"])

    # Tokenization
    df["tokens"] = tokenize_series(df["review"])
    df["tokens"] = filter_single_letters_series(df["tokens"])

    # Spell correction
    correction_map = build_correction_map(df["tokens"])
    df["tokens"] = apply_corrections_series(df["tokens"], correction_map)

    # Stopwords
    df["tokens"] = remove_stopwords_series(df["tokens"])

    # Rare words
    rare_tokens = build_rare_tokens(df["tokens"])
    df["tokens"] = replace_rare_series(df["tokens"], rare_tokens)

    # Final output (UNCHANGED)
    X = df["tokens"]
    y = df["sentiment"]

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    tracemalloc.stop()

    return X, y, df

In [5]:
X, y, df = process_dataframe(df)

normalize_numbers_series:
  Time: 2.09s
  Memory Δ: 40.79 MB
  Peak usage: 102.55 MB

clean_review_series:
  Time: 4.75s
  Memory Δ: 20.98 MB
  Peak usage: 189.65 MB

normalize_emoticons_series:
  Time: 5.99s
  Memory Δ: 3.66 MB
  Peak usage: 172.34 MB

replace_chat_words_series:
  Time: 14.62s
  Memory Δ: 1.33 MB
  Peak usage: 170.09 MB

tokenize_series:
  Time: 22.24s
  Memory Δ: 516.82 MB
  Peak usage: 685.59 MB
  Total tokens: 11788188
  Unique tokens: 101557

filter_single_letters_series:
  Time: 2.24s
  Memory Δ: 101.81 MB
  Peak usage: 787.41 MB
  Total tokens: 11505645
  Unique tokens: 101518

build_correction_map:
  Time: 32.49s
  Memory Δ: 3.84 MB
  Peak usage: 795.25 MB

apply_corrections_series:
  Time: 2.76s
  Memory Δ: 101.81 MB
  Peak usage: 788.84 MB
  Total tokens: 11505645
  Unique tokens: 91940

remove_stopwords_series:
  Time: 12.08s
  Memory Δ: 55.27 MB
  Peak usage: 245.73 MB
  Total tokens: 6037495
  Unique tokens: 91804

build_rare_tokens:
  Time: 6.73s
  Memory

## Self-Attention & Multi-Head Attention — Quick Summary

### 1. Projection Matrices

For each input token vector $X_i \in \mathbb{R}^{d_\text{model}}$, we compute **queries**, **keys**, and **values**:

$$
Q_i = X_i W^Q, \quad
K_i = X_i W^K, \quad
V_i = X_i W^V
$$

- Weight matrices shapes:
$$
W^Q, W^K, W^V \in \mathbb{R}^{d_\text{model} \times d_k}, \quad d_k = \frac{d_\text{model}}{\text{num\_heads}}
$$
- After projection:
$$
Q, K, V \in \mathbb{R}^{\text{seq\_len} \times d_k}
$$

---

### 2. Attention Calculation

1. Compute **attention scores**:
$$
\text{scores} = \frac{Q K^\top}{\sqrt{d_k}}
$$

2. Apply **softmax** to get attention weights:
$$
\alpha = \text{softmax}\Big(\frac{Q K^\top}{\sqrt{d_k}}\Big)
$$

3. Weighted sum of values to get output:
$$
Z = \alpha V
$$

Each $Z_i$ is a context-aware representation of token $X_i$.

---

### 3. Multi-Head Attention

- Use $h$ heads, each with its own projection matrices $W^Q, W^K, W^V$
- Each head focuses on **different aspects** of the input
- Concatenate all head outputs:
$$
Z_\text{concat} = [Z_1; Z_2; \dots; Z_h] \in \mathbb{R}^{\text{seq\_len} \times d_\text{model}}
$$
- Pass through final linear layer:
$$
Z_\text{final} = Z_\text{concat} W^O, \quad W^O \in \mathbb{R}^{d_\text{model} \times d_\text{model}}
$$

---

### 4. Key Takeaways

- $d_\text{model}$ → input embedding dimension  
- $d_k$ → dimension per head ($d_\text{model} / \text{num\_heads}$)  
- Projection weights are sized $(d_\text{model} \times d_k)$  
- Multi-head attention allows **multiple perspectives** simultaneously  
- Scaling by $\sqrt{d_k}$ keeps softmax numerically stable

Conceptually: Each head is a different “lens” — some focus on syntax, some on semantics, etc.

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from itertools import chain
import os

# -----------------------------
# 1. Build Vocabulary
# -----------------------------
class Vocab:
    def __init__(self, tokens_list, min_freq=1, unk_token="<UNK>"):
        all_tokens = list(chain.from_iterable(tokens_list))
        counter = Counter(all_tokens)
        self.unk_token = unk_token
        self.idx_to_token = [unk_token] + [t for t, c in counter.items() if c >= min_freq]
        self.token_to_idx = {t: i for i, t in enumerate(self.idx_to_token)}
    
    def encode(self, tokens):
        return [self.token_to_idx.get(t, self.token_to_idx[self.unk_token]) for t in tokens]
    
    def __len__(self):
        return len(self.idx_to_token)

# Create vocab from X
vocab = Vocab(X.tolist(), min_freq=2)

# -----------------------------
# 2. Dataset
# -----------------------------
class TextDataset(Dataset):
    def __init__(self, X, y, vocab, max_len=50):
        self.X = [vocab.encode(seq)[:max_len] for seq in X]  # truncate
        self.X = [seq + [0]*(max_len - len(seq)) for seq in self.X]  # pad
        self.y = y
        self.X = torch.tensor(self.X, dtype=torch.long)
        self.y = torch.tensor(self.y, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# -----------------------------
# 3. Dataloaders
# -----------------------------
from sklearn.model_selection import train_test_split

X_train_, X_val, y_train_, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_dataset = TextDataset(X_train_, y_train_, vocab)
val_dataset   = TextDataset(X_val, y_val, vocab)

def get_num_workers():
    if torch.cuda.is_available():
        return 1   # GPU → bottleneck is compute, not loading
    else:
        return min(4, os.cpu_count())  # CPU → parallel loading helps

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=get_num_workers(),
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=get_num_workers(),
    pin_memory=torch.cuda.is_available()
)

In [16]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class TransformerClassifier(pl.LightningModule):
    def __init__(self, vocab_size, embed_dim=192, num_heads=6, ff_dim=384,
                 num_layers=2, num_classes=2, max_len=80, lr=3e-4, dropout=0.2,
                 label_smoothing=0.1):
        super().__init__()
        self.save_hyperparameters()

        # Embedding + dropout + positional encoding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embed_dropout = nn.Dropout(dropout)
        self.pos_encoding = PositionalEncoding(embed_dim, max_len)

        # CLS token for classification
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head
        self.fc = nn.Linear(embed_dim, num_classes)
        self.lr = lr
        self.label_smoothing = label_smoothing

    def forward(self, x):
        batch_size = x.size(0)

        # Embed tokens
        x = self.embedding(x)          # (batch, seq, embed)
        x = self.embed_dropout(x)

        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Add positional encoding
        x = self.pos_encoding(x)

        # Transformer
        x = self.transformer(x)

        # Take CLS token for classification
        x = x[:, 0]
        x = self.fc(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y, label_smoothing=self.label_smoothing)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=self.lr)
        scheduler = {
            'scheduler': optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=1
            ),
            'monitor': 'val_acc',
            'interval': 'epoch',
            'frequency': 1
        }
        return [optimizer], [scheduler]

In [17]:
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning import Trainer

model = TransformerClassifier(vocab_size=len(vocab))


# Define early stopping callback
early_stop_callback = EarlyStopping(
    monitor="val_acc",   # metric to monitor
    min_delta=0.001,     # minimum change to qualify as improvement
    patience=5,          # number of epochs with no improvement to wait
    verbose=True,
    mode="max"           # we want to maximize validation accuracy
)

# Trainer
trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    callbacks=[early_stop_callback],
    log_every_n_steps=10
)
 
trainer.fit(model, train_loader, val_loader) 

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding     │ Embedding          │  5.4 M │ train │     0 │
│ 1 │ embed_dropout │ Dropout            │      0 │ train │     0 │
│ 2 │ pos_encoding  │ PositionalEncoding │      0 │ train │     0 │
│ 3 │ transformer   │ TransformerEncoder │  594 K │ train │     0 │
│ 4 │ fc            │ Linear             │    386 │ train │     0 │
│   │ other params  │ n/a                │    192 │ n/a   │   n/a │
└───┴───────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 6.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.0 M                                                                                                
Total estimated model params size (MB): 24                                                                         
Modules in train mode: 26                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_acc improved. New best score: 0.774
Metric val_acc improved by 0.011 >= min_delta = 0.001. New best score: 0.785
Metric val_acc improved by 0.017 >= min_delta = 0.001. New best score: 0.801
Metric val_acc improved by 0.008 >= min_delta = 0.001. New best score: 0.810
Metric val_acc improved by 0.008 >= min_delta = 0.001. New best score: 0.818
Metric val_acc improved by 0.003 >= min_delta = 0.001. New best score: 0.821
Metric val_acc improved by 0.001 >= min_delta = 0.001. New best score: 0.823
Metric val_acc improved by 0.006 >= min_delta = 0.001. New best score: 0.828
Monitored metric val_acc did not improve in the last 5 records. Best score: 0.828. Signaling Trainer to stop.


In [18]:
!pip install transformers sentencepiece --quiet

In [30]:
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Example: Your data
X = df["review"]  # raw text
y = df["sentiment"]
encoder = LabelEncoder()
y = encoder.fit_transform(y)

# ✅ 1️⃣ Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ✅ 2️⃣ Lightning Module
class BertClassifier(pl.LightningModule):
    def __init__(self, n_classes=2, lr=2e-5):
        super().__init__()
        self.save_hyperparameters()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            "prajjwal1/bert-tiny",
            num_labels=n_classes
        )

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self(**batch)
        loss = outputs.loss
        logits = outputs.logits
        acc = (logits.argmax(dim=1) == batch['labels']).float().mean()
        self.log('train_loss', loss)
        self.log('train_acc', acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(**batch)
        loss = outputs.loss
        logits = outputs.logits
        acc = (logits.argmax(dim=1) == batch['labels']).float().mean()
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=self.hparams.lr)

# ✅ 3️⃣ Tokenizer
tokenizer =AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")

In [31]:
# 3️⃣ Prepare data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df["review"]
y = df["sentiment"]
encoder = LabelEncoder()
y = encoder.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_dataset = TextDataset(X_train.values, y_train, tokenizer, max_len=128)
val_dataset   = TextDataset(X_val.values, y_val, tokenizer, max_len=128)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=3)
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=3)

In [32]:
# 4️⃣ Training
from pytorch_lightning import Trainer

# ✅ 6️⃣ Model
model = BertClassifier(n_classes=2, lr=2e-5)


early_stop_callback = EarlyStopping(
    monitor="val_acc",   # metric to monitor
    min_delta=0.001,     # minimum change to qualify as improvement
    patience=5,          # number of epochs with no improvement to wait
    verbose=True,
    mode="max"           # we want to maximize validation accuracy
)



trainer = Trainer(
    max_epochs=20,
    accelerator='auto',
    devices=1,
    log_every_n_steps=10,
    callbacks=[early_stop_callback],
)

trainer.fit(model, train_loader, val_loader)

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                          ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ BertForSequenceClassification │  4.4 M │ eval │     0 │
└───┴───────┴───────────────────────────────┴────────┴──────┴───────┘

Trainable params: 4.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.4 M                                                                                                
Total estimated model params size (MB): 17                                                                         
Modules in train mode: 0                                                                                           
Modules in eval mode: 51                                                                                           
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 51 module(s) in eval mode at
the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore
this warning.

Metric val_acc improved. New best score: 0.808
Metric val_acc improved by 0.022 >= min_delta = 0.001. New best score: 0.830
Metric val_acc improved by 0.004 >= min_delta = 0.001. New best score: 0.834
Monitored metric val_acc did not improve in the last 5 records. Best score: 0.834. Signaling Trainer to stop.
